In [29]:
import pandas as pd
import numpy as np
import sqlite3
from sklearn.linear_model import LogisticRegression

conn = sqlite3.connect("../fraud.db")
X_train = pd.read_sql(""" SELECT * FROM X_train""", conn)
X_test = pd.read_sql(""" SELECT * FROM X_test""", conn)
y_train = pd.read_sql(""" SELECT * FROM y_train""", conn)['Class']
y_test = pd.read_sql(""" SELECT * FROM y_test""", conn)['Class']
conn.close()

In [30]:
model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000).fit(X_train, y_train)

In [31]:
predictions = model.predict(X_test)
probs = model.predict_proba(X_test)[:, 1]

print(predictions[:5])
print(y_test[:5])

print(probs)

[0 0 0 0 1]
0    0
1    0
2    0
3    0
4    0
Name: Class, dtype: int64
[0.00584217 0.06918075 0.00013404 ... 0.00027176 0.00392062 0.08460862]


In [32]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

report = classification_report(y_test, predictions, output_dict=True)
conf_mat = confusion_matrix(y_test, predictions)
roc_score = roc_auc_score(y_test, probs)
pr_score = average_precision_score(y_test, probs)

print(f"CLASSIFICATION REPORT: {report['1']}")
print(f"CONFUSION MATRIX: {conf_mat}")
print(f"AUC-ROC:  {roc_score:.4f}")
print(f"AUC-PR:   {pr_score:.4f}")

CLASSIFICATION REPORT: {'precision': 0.06097560975609756, 'recall': 0.9183673469387755, 'f1-score': 0.11435832274459974, 'support': 98.0}
CONFUSION MATRIX: [[55478  1386]
 [    8    90]]
AUC-ROC:  0.9722
AUC-PR:   0.7159


In [33]:
results = pd.DataFrame([{
    'model': "Logistic Regression",
    'recall': report['1']['recall'],
    'precision': report['1']['precision'],
    'f1': report['1']['f1-score'],
    'auc_roc' : roc_score,
    'auc_pr': pr_score
}])

conn = sqlite3.connect("../fraud.db")
results.to_sql('model_results', conn, if_exists='replace', index=False)
conn.close()

### Review of results from Logistic Regression Model

The Logistic Regression model proved to be a strong baseline for our fraud detection. It achieved a 0.918 Recall score and a 0.972 AUC-ROC score. However, where it failed was in the precision, generating a significant amount of false positives. The High AUC-PR 0.716, shows that a meaningful signal exists in the features. Next, we'll implmement a non-linear model, in hopes of improving the precision-recall balance